# EPCLM-004 M5 — BMOJO-inspired Innovation Store
Tesla T4 + Internet + existing GITHUB_TOKEN and KAGGLE_API_TOKEN secrets. One Save & Run All: Gate 0 archives M4 Version 187 without retraining → Gates 1–6 → five fresh E3/E6/E7 engineering controls, two FP32 updates each → capacity/task/NL diagnostics → verified publication. No M6/20M/100M.

In [ ]:
from pathlib import Path
import os, signal, subprocess, sys, time
started=time.monotonic()
hard_deadline=started+7200
repo=Path('/kaggle/working/minicells-epclm004-m5')
branch='research/epclm-004-m5-innovation-store'

def run(command,cwd=None,limit=180,stop_at=None,log=None):
    remaining=min(limit,(stop_at or hard_deadline)-time.monotonic())
    if remaining<=0:raise TimeoutError('M5 budget exhausted; Output retained')
    stream=open(log,'a',buffering=1) if log else None
    try:
        p=subprocess.Popen(command,cwd=cwd,start_new_session=True,
                           stdout=stream,stderr=subprocess.STDOUT if stream else None)
        try:code=p.wait(timeout=remaining)
        except subprocess.TimeoutExpired:
            os.killpg(p.pid,signal.SIGTERM)
            try:p.wait(timeout=15)
            except subprocess.TimeoutExpired:os.killpg(p.pid,signal.SIGKILL);p.wait()
            raise TimeoutError('M5 stopped at budget boundary; inspect retained Output')
    finally:
        if stream:stream.close()
    if log:print(Path(log).read_text(),flush=True)
    if code:raise RuntimeError('M5 gate failed; inspect decision.json and raw.log in Output')

if not repo.exists():
    run(['git','clone','--depth=1','--single-branch','--branch',branch,'https://github.com/ArcheLabs/mini-cells.git',str(repo)])
else:
    origin=subprocess.check_output(['git','-C',str(repo),'remote','get-url','origin'],text=True).strip()
    if origin.removesuffix('.git')!='https://github.com/ArcheLabs/mini-cells':raise ValueError('unrelated checkout')
    run(['git','-C',str(repo),'fetch','--depth=1','origin',branch])
    run(['git','-C',str(repo),'switch','--detach','FETCH_HEAD'])
source=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
base=repo/'research/projects/cellular-language-models/experiments/epclm-004-unified-memory-10m/m5'
out=Path('/kaggle/working/epclm004-m5-'+source[:12]);out.mkdir(exist_ok=True)
run([sys.executable,'-m','pip','install','-q','-r',str(base/'requirements.txt')],limit=300)
# Install only missing WikiText adapters; do not upgrade the entire environment.
import importlib.util
missing=[pin for name,pin in [('datasets','datasets==3.6.0'),('tokenizers','tokenizers==0.20.3')] if importlib.util.find_spec(name) is None]
if missing:run([sys.executable,'-m','pip','install','-q',*missing],limit=300)
os.environ['OMP_NUM_THREADS']='1';os.environ['PYTEST_DISABLE_PLUGIN_AUTOLOAD']='1'
# One invocation: ordered cached CPU gates, then actual T4 Smoke and verified publication.
# No duplicate pytest invocation; tests and entry point share the same gate functions.
run([sys.executable,str(base/'run.py'),'--output',str(out),'--started',str(started),'--publish'],
    cwd=repo,limit=6600,stop_at=started+6600,log=out/'raw.log')
